[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aai-services/datainsight-curriculum/blob/main/stages/stage-1/1.6-uncertainty-by-simulation/lesson.ipynb)

# 1.6 Uncertainty by simulation

Week 9.

By the end of this lesson you can:

- Explain why a number computed from a sample would change if you took another sample
- Use the bootstrap to estimate how much it would change
- Report an estimate with a 95% confidence interval, and interpret the interval correctly

We use simulation instead of formulas: the ideas are the same, and you can see every step.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = "https://raw.githubusercontent.com/aai-services/datainsight-curriculum/main/datasets/"

countries = pd.read_csv(DATA + "countries.csv")
penguins = pd.read_csv(DATA + "penguins.csv")

rng = np.random.default_rng(42)   # a fixed seed makes the random results repeatable

## Samples vary

Suppose the life expectancies of all countries in 2020 are the whole **population** we care about. In real work we rarely see the whole population; we see a **sample**. Let us draw a few random samples of 20 countries and compute each sample's mean.

In [ ]:
population = countries.loc[countries["year"] == 2020, "life_expectancy"].dropna().to_numpy()
print(f"Population: {len(population)} countries, true mean {population.mean():.2f} years")

for i in range(5):
    sample = rng.choice(population, size=20, replace=False)
    print(f"Sample {i + 1}: mean {sample.mean():.2f}")

Each sample gives a different answer. That variation is not a mistake: it is what happens whenever we learn from part of a population.

## The sampling distribution

Repeat the sampling many times and plot all the sample means. The spread of this distribution tells us how far a single sample mean is likely to be from the truth. Its standard deviation is called the **standard error**.

In [ ]:
def sample_means(values, n, repeats, rng):
    return np.array([rng.choice(values, size=n, replace=False).mean() for _ in range(repeats)])

means_20 = sample_means(population, 20, 2000, rng)
means_80 = sample_means(population, 80, 2000, rng)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(means_20, bins=40, alpha=0.6, label="samples of 20")
ax.hist(means_80, bins=40, alpha=0.6, label="samples of 80")
ax.axvline(population.mean(), color="black", linestyle="--", label="true mean")
ax.set_title("Means of 2,000 random samples")
ax.set_xlabel("Sample mean life expectancy (years)")
ax.set_ylabel("Number of samples")
ax.legend()
plt.show()

print(f"Standard error with n = 20: {means_20.std():.2f} years")
print(f"Standard error with n = 80: {means_80.std():.2f} years")

Larger samples give means that cluster more tightly around the truth.

## The bootstrap: one sample is all we have

In practice we cannot draw thousands of samples. The **bootstrap** uses the one sample we have as a stand-in for the population: it resamples from our sample **with replacement**, many times, and looks at how much the statistic varies.

In [ ]:
def bootstrap_means(values, n_boot, rng):
    """Means of n_boot resamples, each the same size as values, drawn with replacement."""
    values = np.asarray(values)
    resamples = rng.choice(values, size=(n_boot, len(values)), replace=True)
    return resamples.mean(axis=1)

our_sample = rng.choice(population, size=40, replace=False)
boot = bootstrap_means(our_sample, 5000, rng)

print(f"Our sample mean: {our_sample.mean():.2f}")
print(f"Bootstrap estimate of the standard error: {boot.std():.2f}")

## A 95% confidence interval

The middle 95% of the bootstrap means, from the 2.5th to the 97.5th percentile, gives a **95% confidence interval**.

In [ ]:
low, high = np.percentile(boot, [2.5, 97.5])
print(f"Mean life expectancy: {our_sample.mean():.1f} years (95% CI {low:.1f} to {high:.1f})")
print(f"True population mean: {population.mean():.1f}")

### What the interval means

The method is designed so that, if we repeated the whole process (sample, then bootstrap) many times, about 95% of the intervals would contain the true mean. We can check that claim here, because we know the truth:

In [ ]:
hits = 0
trials = 400
for _ in range(trials):
    s = rng.choice(population, size=40, replace=False)
    lo, hi = np.percentile(bootstrap_means(s, 1000, rng), [2.5, 97.5])
    hits += lo <= population.mean() <= hi
print(f"{hits / trials:.0%} of {trials} intervals contained the true mean")

The result is close to 95%. With small samples the simple percentile bootstrap tends to fall a little short; treat it as approximate below about 20 observations.

**Common mistakes when reporting an interval:**

- "There is a 95% chance the true mean is in this interval." The true mean is fixed; it is the interval that varies from sample to sample. Say instead: "We are 95% confident the interval contains the true mean", understood as a statement about the method.
- "95% of the data lie in this interval." No: the interval is about the **mean**, not individual values.
- Ignoring how the sample was collected. The bootstrap only measures random sampling variation. If the sample is biased, for example only countries with good records, the interval does not fix that.

## Comparing two groups

The same idea works for a **difference**. Resample each group separately, compute the difference in means each time, and take the middle 95%.

In [ ]:
adelie = penguins.loc[penguins["species"] == "Adelie", "flipper_length_mm"].dropna().to_numpy()
gentoo = penguins.loc[penguins["species"] == "Gentoo", "flipper_length_mm"].dropna().to_numpy()

observed = gentoo.mean() - adelie.mean()
diffs = bootstrap_means(gentoo, 5000, rng) - bootstrap_means(adelie, 5000, rng)
low, high = np.percentile(diffs, [2.5, 97.5])

print(f"Gentoo minus Adelie flipper length: {observed:.1f} mm (95% CI {low:.1f} to {high:.1f})")

## Worked example: reporting a difference

Do male and female Adelie penguins differ in body mass, and by how much?

In [ ]:
adelie_all = penguins[penguins["species"] == "Adelie"].dropna(subset=["sex", "body_mass_g"])
male = adelie_all.loc[adelie_all["sex"] == "male", "body_mass_g"].to_numpy()
female = adelie_all.loc[adelie_all["sex"] == "female", "body_mass_g"].to_numpy()

diff = male.mean() - female.mean()
boot_diff = bootstrap_means(male, 5000, rng) - bootstrap_means(female, 5000, rng)
low, high = np.percentile(boot_diff, [2.5, 97.5])

print(f"n = {len(male)} male, {len(female)} female")
print(f"Male minus female: {diff:.0f} g (95% CI {low:.0f} to {high:.0f})")

**A written report:**

> Among Adelie penguins with a recorded sex (73 male, 73 female), males were on average about 675 g heavier than females (95% CI roughly 580 to 780 g; bootstrap, 5,000 resamples). The whole interval is well above zero, so the data are consistent with a real difference of at least about half a kilogram. The sample covers three islands between 2007 and 2009, so the result may not apply to other colonies or years.

This is the standard for Project 2: the size of the difference, its uncertainty, how it was computed, and the limits of what the data can say.

## Your turn

Open `practice-1.6.ipynb`.